# Tutorial 2 -- Multiphoton STAR Simulations with Arbitrary Excitation Order and Replica

This tutorial repeats the simulation workflow but with arbitrary excitation order and reconstruction from different multiphoton replicas. Change `n`, `m`, or `longitudinal_mode` below to explore another analytical sequence.

In [ ]:
from pathlib import Path
import sys

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'src' / 'multiphoton_star').is_dir()),
    None,
)
if repo_root is None:
    raise RuntimeError('Run this notebook from within the multiphoton-star-toolbox repository.')
sys.path.insert(0, str(repo_root / 'src'))

import matplotlib.pyplot as plt
import numpy as np
from multiphoton_star import phantoms, plotting, pulse, radial, recon, simulate

## 1. Pulse sequence and simulation parameters

`n=0` is the separate one-photon path. For nonzero `n`, the photon terminology is $|n|+1$ photons. `m` selects a received spectral replica and does not change the transmitted waveform. Here, we choose n=-2 which corresponds to three-photon excitation, but we reconstruct from m=+1 (the first-order replica).

When $f_z$ is not large relative to `bw_hz`, neighboring replicas can overlap and the model approximations become less accurate, so reconstruction error increases, particularly for phase-modulated $B_{1,z}$. This error typically grows with $|n|$: the default modulation index $a$ (the first maximum of $|J_n(a)|$) increases with $|n|$, which strengthens the phase terms neglected by the approximation. Increasing $f_z$ reduces this error but requires a smaller `dt_s`.

In [ ]:
n = -2
m = +1
longitudinal_mode = 'constant'  # or 'phase_modulated'

beta = 5.3
q = 2
bw_hz = 1_000.0
R = 200.0
B1xy_max_G = 5e-4
f_z_hz = 3 * bw_hz  # f_z/bw = 3
dt_s = 1.0 / (11 * max(abs(n), 1) * f_z_hz)  # ~11 samples per fastest field period (|n| f_z)
n_workers = 3       # Parallel simulation workers (~5 GB memory each for n=-2)
FOV_cm = (4.0, 4.0)
phantom_Ny = 200
phantom_Nx = 200

samples_per_f_z_period = 1.0 / (f_z_hz * dt_s)
print(f'Samples per f_z period: {samples_per_f_z_period:.1f}')

## 2. Construct multiphoton transmit waveforms

The reference pulse is converted into $B_{1,xy}$ and $B_{1,z}$ fields for the selected excitation order $n$. The $B_{1,xy}$ carrier is shifted in frequency by $nf_z$. In constant-$B_{1,z}$ mode, $B_{1,xy}$ keeps the HS sweep and $B_{1,z}$ is a sinusoid at $f_z$; in phase-modulated mode, $B_{1,xy}$ is unswept and the sweep moves into the phase of $B_{1,z}(t)=B_{1,z}\cos(\omega_z t+\phi(t)/n)$.

By default, the modulation index $a=\gamma B_{1,z}/\omega_z$ is set to the first maximum of $|J_n(a)|$, and the $B_{1,xy}$ amplitude is multiplied by $1/|J_n(a)|$. This compensates for the Bessel scaling of the multiphoton excitation, so the one-photon flip-angle estimate also applies to the multiphoton pulse (small-flip approximation). Because the maximum of $|J_n(a)|$ decreases with $|n|$, higher orders require a larger $B_{1,xy}$ scale and therefore more transmit power.

These defaults can be changed in `make_multiphoton`:
- `a`: use a different modulation index. $B_{1,xy}$ is still scaled by $1/|J_n(a)|$, so the flip angle remains matched.
- `B1xy_scale` or `B1xy_override_G`: set the transverse amplitude or the complete $B_{1,xy}$ waveform directly. This disables the automatic flip-angle matching.
- `B1z_override_G`: supply a custom $B_{1,z}$ waveform. A general waveform has no single-Bessel scaling, so `B1xy_scale` or `B1xy_override_G` must also be given.

In [ ]:
# One-photon reference pulse used to construct the multiphoton waveforms.
reference = pulse.generate_hs_pulse(
    beta=beta, q=q, bw_hz=bw_hz, R=R, dt_s=dt_s,
    B1xy_max_G=B1xy_max_G, verbose=False,
)
fields = reference.make_multiphoton(
    n=n, f_z_Hz=f_z_hz, mode=longitudinal_mode, verbose=False,
)
print(f'n={fields.n:+d}, reconstruction m={m:+d}, mode={fields.mode}')
print(f'a={fields.a:.6f}, J_n(a)={fields.excitation_coefficient:.6f}')
print(f'B1xy scale={fields.B1xy_scale:.6f}, B1z amplitude={fields.B1z_amplitude_G:.6f} G')
print(f'Estimated flip angle: {reference.flip_angle_deg:.2f} degrees')

In [ ]:
figure, _ = plotting.plot_waveforms(
    reference.t_s, reference.B1xy_G, domain='both', normalize_spectra=True
);
figure.suptitle('One-photon HS pulse');

In [ ]:
figure, _ = plotting.plot_waveforms(
    reference.t_s, fields.B1xy_G, fields.B1z_G,
    domain='both', normalize_spectra=True,
);
figure.suptitle(f'Multiphoton fields: n={n:+d}, {fields.mode}');

## 3. Run the Bloch simulator

We simulate both the one-photon reference and the selected multiphoton sequence with the included modified Bloch simulator. As in the original Shepp--Logan workflow, the simulation object is explicitly sampled on a 200 x 200 grid, while resolution, output matrix, and spoke count are derived from the pulse and FOV. 

In [ ]:
trajectory = radial.generate_radial_gradients(
    bw_hz=bw_hz, FOV_cm=FOV_cm, pulse=reference,
    verbose=False,
)
resolution_cm = trajectory.resolution_cm
reconstruction_Nx = int(round(FOV_cm[0] / resolution_cm))
reconstruction_Ny = int(round(FOV_cm[1] / resolution_cm))
obj = phantoms.shepp_logan_bloch(
    Ny=phantom_Ny, Nx=phantom_Nx, FOV_cm=FOV_cm,
    threshold=1e-6,
)
expected_spokes = int(np.ceil(np.pi * min(FOV_cm) / resolution_cm))
plt.figure(figsize=(4, 4))
plt.imshow(obj.img, cmap='gray', origin='lower', extent=(-2, 2, -2, 2))
plt.xlabel('x (cm)')
plt.ylabel('y (cm)')
plt.title(f'{phantom_Ny}x{phantom_Nx} simulation object');
print(f'Simulation object: {phantom_Ny}x{phantom_Nx}')
print(f'Reconstruction: {reconstruction_Ny}x{reconstruction_Nx}')
print(f'{trajectory.N_spokes} radial spokes')
print(f'Trajectory resolution: {trajectory.resolution_cm:.3f} cm')
assert phantom_Nx >= 2 * reconstruction_Nx
assert phantom_Ny >= 2 * reconstruction_Ny
assert trajectory.N_spokes == expected_spokes

In [ ]:
%%time
reference_sim = simulate.simulate_radial_bloch(
    pulse=reference, traj=trajectory, positions_cm=obj.positions_cm,
    mx0=obj.mx0, my0=obj.my0, mz0=obj.mz0,
    workers=n_workers, progress=True, tqdm_desc='One-photon reference',
)

In [ ]:
%%time
multiphoton_sim = simulate.simulate_radial_bloch(
    pulse=reference, traj=trajectory, positions_cm=obj.positions_cm,
    mx0=obj.mx0, my0=obj.my0, mz0=obj.mz0,
    workers=n_workers, progress=True, tqdm_desc=f'n={n:+d} multiphoton',
    **fields.to_sim_kwargs(),
)

In [ ]:
spoke_index = 0

reference_figure, _ = plotting.plot_spoke(
    reference_sim, spoke_index=spoke_index,
)
reference_figure.suptitle(
    f'One-photon reference: raw spoke {spoke_index}'
)

multiphoton_figure, _ = plotting.plot_spoke(
    multiphoton_sim, spoke_index=spoke_index,
)
multiphoton_figure.suptitle(
    f'n={n:+d} multiphoton: raw spoke {spoke_index}'
)

## 4. Reconstruct and compare images

For phase-modulated fields, the nominal window width is $(1+|m/n|)\,\mathrm{bw}$; the default practical guard multiplies that total width by 1.1. Constant-$B_{1,z}$ uses $1.1\,\mathrm{bw}$.

In [ ]:
reference_spokes = recon.deconvolve_with_pulse(reference_sim.raw_mat, reference)
selected_spokes = recon.deconvolve_replica(
    raw_mat=multiphoton_sim.raw_mat, pulse=reference, waveforms=fields,
    m=m, bw_Hz=bw_hz,
)

In [ ]:
reference_image = recon.reconstruct_radial_image_2d(
    reference_spokes.x_time, trajectory, FOV_cm=FOV_cm,
    dcf='ramp', edge_taper_r0=0.85, edge_taper_kind='tukey',
    eps=1e-6, grid_oversamp=2.0,
)
selected_image = recon.reconstruct_radial_image_2d(
    selected_spokes.x_time, trajectory, FOV_cm=FOV_cm,
    dcf='ramp', edge_taper_r0=0.85, edge_taper_kind='tukey',
    eps=1e-6, grid_oversamp=2.0,
)

In [ ]:
# Display images (masked)
reference_array = reference_image.F_cart
selected = selected_image.F_cart
mask = np.abs(reference_array) >= 0.10 * np.max(np.abs(reference_array))
for label, image in [('One photon', reference_array), (f'n={n:+d}, m={m:+d}', selected)]:
    figure, _ = plotting.plot_image(
        image, display='both', mask=mask, normalize_magnitude=True,
        extent_cm=(-2, 2, -2, 2),
    )
    figure.suptitle(label)

In [ ]:
magnitude_nrmse = (
    np.linalg.norm((np.abs(selected) - np.abs(reference_array))[mask])
    / np.linalg.norm(np.abs(reference_array)[mask])
)
phase_difference = np.angle(selected[mask] * np.conj(reference_array[mask]))
phase_rms_degrees = np.degrees(np.sqrt(np.mean(phase_difference**2)))
print(f'Magnitude NRMSE: {magnitude_nrmse:.4f}')
print(f'Phase RMS: {phase_rms_degrees:.2f} degrees')